## Code for Wise Case
[Job Position](https://wise.jobs/job/senior-product-analyst-regional-expansion-in-sao-paulo-jid-1656)

##### Bootstrap the data

In [57]:
import pandas as pd
import numpy as np

file_path = "../data/wise_funnel_events_regional_v2.csv"
df = pd.read_csv(file_path)

##### EDA

In [62]:
# Understand schema, missing values, date coverage, and the main categorical values
# before calculating metrics.

# Copy to avoid accidental mutation of raw data
raw_df = df.copy()

raw_df.info()
raw_df.describe(include='all')

<class 'pandas.DataFrame'>
RangeIndex: 73440 entries, 0 to 73439
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   event_name       73440 non-null  str           
 1   dt               73440 non-null  datetime64[us]
 2   source_currency  73440 non-null  str           
 3   target_currency  73440 non-null  str           
 4   user_id          73440 non-null  int64         
 5   user_country     73440 non-null  str           
 6   platform         73440 non-null  str           
 7   experience       73440 non-null  str           
 8   month            73440 non-null  str           
dtypes: datetime64[us](1), int64(1), str(7)
memory usage: 5.0 MB


,event_name,dt,source_currency,target_currency,user_id,user_country,platform,experience,month
count,73440,73440,73440,73440,7.344000e+04,73440,73440,73440,73440
unique,3,NaN,1,1,NaN,3,3,2,3
top,Transfer Created,NaN,MXN,USD,NaN,USA,Android,New,2024-02
freq,43070,NaN,73440,73440,NaN,28307,27593,39812,42634
mean,NaN,2024-02-03 18:36:20,NaN,NaN,1.501395e+06,NaN,NaN,NaN,NaN
min,NaN,2024-01-01 00:00:00,NaN,NaN,1.000006e+06,NaN,NaN,NaN,NaN
25%,NaN,2024-01-22 00:00:00,NaN,NaN,1.251646e+06,NaN,NaN,NaN,NaN
50%,NaN,2024-02-05 00:00:00,NaN,NaN,1.500739e+06,NaN,NaN,NaN,NaN
75%,NaN,2024-02-17 00:00:00,NaN,NaN,1.754569e+06,NaN,NaN,NaN,NaN
max,NaN,2024-03-01 00:00:00,NaN,NaN,1.999991e+06,NaN,NaN,NaN,NaN


#### How many users have more than one country?

In [58]:
country_combos = (
    df.groupby('user_id')['user_country']
    .agg(lambda x: ' + '.join(sorted(x.unique())))
    .reset_index(name='countries')
)

stage_counts = (
    df.groupby(['user_id', 'event_name'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

result = country_combos.merge(stage_counts, on='user_id')

summary = (
    result.groupby('countries')
    .agg(
        unique_users=('user_id', 'nunique'),
        Transfer_Created=('Transfer Created', 'sum'),
        Transfer_Funded=('Transfer Funded', 'sum'),
        Transfer_Transferred=('Transfer Transferred', 'sum'),
    )
    .rename(columns={
        'unique_users': '# Users',
        'Transfer_Created': 'Transfer Created',
        'Transfer_Funded': 'Transfer Funded',
        'Transfer_Transferred': 'Transfer Transferred',
    })
    .sort_values('# Users', ascending=False)
)

summary['Total'] = summary[['Transfer Created', 'Transfer Funded', 'Transfer Transferred']].sum(axis=1)

summary.loc['Total'] = summary.sum()

# Percentage of column total
cols = ['# Users', 'Transfer Created', 'Transfer Funded', 'Transfer Transferred', 'Total']
for col in cols:
    summary[f'{col} %'] = (summary[col] / summary[col].sum() * 100).round(1).astype(str) + '%'

summary.loc['Total'] = summary[cols].sum()

summary

,# Users,Transfer Created,Transfer Funded,Transfer Transferred,Total,# Users %,Transfer Created %,Transfer Funded %,Transfer Transferred %,Total %
countries,,,,,,,,,,
USA,15276,15983,7957,3539,27479,19.0%,18.6%,20.6%,16.0%,18.7%
MEX,15270,15865,5145,3961,24971,19.0%,18.4%,13.3%,17.9%,17.0%
OTHER,9110,9997,5623,3266,18886,11.3%,11.6%,14.6%,14.7%,12.9%
MEX + USA,284,600,269,155,1024,0.4%,0.7%,0.7%,0.7%,0.7%
OTHER + USA,142,319,169,82,570,0.2%,0.4%,0.4%,0.4%,0.4%
MEX + OTHER,139,298,124,75,497,0.2%,0.3%,0.3%,0.3%,0.3%
MEX + OTHER + USA,2,8,3,2,13,0.0%,0.0%,0.0%,0.0%,0.0%
Total,80446,86140,38580,22160,146880,NaN,NaN,NaN,NaN,NaN


In [ ]:
# CleanUp Duplicated Countries
# build combination per user_id
combo_map = (
    df.groupby('user_id')['user_country']
    .agg(lambda x: ' + '.join(sorted(x.unique())))
)

# resolve ambiguous combos to the known real country
COMBO_RESOLUTION = {
    'OTHER + USA': 'USA',
    'MEX + OTHER': 'MEX',
}

resolved_lookup = {
    user_id: COMBO_RESOLUTION[combo]
    for user_id, combo in combo_map.items()
    if combo in COMBO_RESOLUTION
}

def xlookup_country(user_id, user_country):
    """Return resolved country for ambiguous users, original country otherwise."""
    return resolved_lookup.get(user_id, user_country)

df['resolved_country'] = df.apply(lambda r: xlookup_country(r['user_id'], r['user_country']), axis=1)

# sanity check
print("Before:", df['user_country'].value_counts().to_dict())
print("After: ", df['resolved_country'].value_counts().to_dict())
df.describe(include='all')

In [ ]:
monthly = (
    df.assign(month=pd.to_datetime(df['dt']).dt.to_period('M').astype(str))
    .groupby(['month', 'event_name'])
    .size()
    .unstack(fill_value=0)
    .rename(columns={'Transfer Created': 'created', 'Transfer Funded': 'funded', 'Transfer Transferred': 'transferred'})
    .reset_index()
)

monthly['funded_rate']      = (monthly['funded']      / monthly['created']).round(3)
monthly['end_to_end_rate']  = (monthly['transferred'] / monthly['created']).round(3)

monthly[['month', 'created', 'funded', 'transferred', 'funded_rate', 'end_to_end_rate']]

In [ ]:
df['month'] = pd.to_datetime(df['dt']).dt.to_period('M').astype(str)

# --- Base counts per month ---
base = (
    df.groupby(['month', 'event_name'])
    .size()
    .unstack(fill_value=0)
    .rename(columns={'Transfer Created': 'created', 'Transfer Funded': 'funded', 'Transfer Transferred': 'transferred'})
)

# --- Funded Rate & End-to-End Conversion ---
base['funded_rate']     = base['funded']      / base['created']
base['end_to_end_rate'] = base['transferred'] / base['created']

# --- MoM Growth (Created) ---
base['mom_growth'] = base['created'].pct_change()

# --- New User Funded Rate ---
new_users = (
    df[df['experience'] == 'New']
    .groupby(['month', 'event_name']).size()
    .unstack(fill_value=0)
)
base['new_funded_rate'] = new_users['Transfer Funded'] / new_users['Transfer Created']

# --- Existing User Funded Rate ---
existing_users = (
    df[df['experience'] == 'Existing']
    .groupby(['month', 'event_name']).size()
    .unstack(fill_value=0)
)
base['existing_funded_rate'] = existing_users['Transfer Funded'] / existing_users['Transfer Created']

# --- New vs Existing Gap ---
base['new_vs_existing_gap'] = base['existing_funded_rate'] - base['new_funded_rate']

# --- Platform Funded Rate (one column per platform) ---
platform = (
    df.groupby(['month', 'platform', 'event_name']).size()
    .unstack(fill_value=0)
)
for p in df['platform'].unique():
    col = f'funded_rate_{p.lower()}'
    base[col] = platform.loc[(slice(None), p), 'Transfer Funded'].values / platform.loc[(slice(None), p), 'Transfer Created'].values

base.round(3)

### Charts

In [ ]:
import pandas as pd


df = pd.read_csv('../data/wise_funnel_events_regional_v2.csv')
df['dt'] = pd.to_datetime(df['dt'])
df['week'] = df['dt'] - pd.to_timedelta(df['dt'].dt.dayofweek, unit='D')  # floor to Monday

weekly = df.groupby(['week', 'event_name']).size().reset_index(name='count')

funnel_order = ['Transfer Created', 'Transfer Funded', 'Transfer Transferred']

fig = px.line(
    weekly, x='week', y='count',
    color='event_name', category_orders={'event_name': funnel_order},
    markers=True,
    title='Weekly Transfer Volume by Funnel Stage',
    labels={'week': 'Week', 'count': 'Events', 'event_name': 'Funnel Stage'}
)
fig.update_xaxes(dtick=5 * 24 * 60 * 60 * 1000, tick0='-2024-Jan-01')

fig.show()

In [ ]:
import pandas as pd
import plotly.express as px

df = pd.read_csv('../data/wise_funnel_events_regional_v2.csv')
df['dt'] = pd.to_datetime(df['dt'])

df['period'] = 'Full Period'
df.loc[df['dt'].dt.month == 1, 'period'] = 'January'
df.loc[df['dt'].dt.month == 2, 'period'] = 'February'

funnel_order = ['Transfer Created', 'Transfer Funded', 'Transfer Transferred']
period_order = ['Full Period', 'January', 'February']

funnel_df = (
    df.groupby(['period', 'experience', 'event_name'])
    .size()
    .reset_index(name='count')
)
funnel_df['event_name'] = pd.Categorical(funnel_df['event_name'], categories=funnel_order, ordered=True)
funnel_df = funnel_df.sort_values(['period', 'event_name'])

created = (
    funnel_df[funnel_df['event_name'] == 'Transfer Created']
    [['period', 'experience', 'count']]
    .rename(columns={'count': 'created'})
)
funnel_df = funnel_df.merge(created, on=['period', 'experience'])
funnel_df['label'] = funnel_df.apply(lambda r: f"{r['count']:,} ({r['count']/r['created']:.0%})", axis=1)

fig = px.funnel(
    funnel_df, y='event_name', x='count',
    color='experience', text='label',
    facet_col='period',
    category_orders={'event_name': funnel_order, 'period': period_order},
    title='Transfer Funnel Conversion by User Experience',
    labels={'event_name': 'Funnel Stage', 'count': 'Users', 'experience': 'Experience'}
)

fig.show()

In [56]:
# Converting the date field to month
df['month'] = pd.to_datetime(df['dt']).dt.to_period('M').astype(str)

# Base counts of events per month
base = (
    df.groupby(['month', 'event_name'])
    .size()
    .unstack(fill_value=0)
    .rename(columns={'Transfer Created': 'created', 'Transfer Funded': 'funded', 'Transfer Transferred': 'transferred'})
)

# --- Funded Rate & End-to-End Conversion ---
base['funded_rate']     = base['funded']      / base['created']
base['end_to_end_rate'] = base['transferred'] / base['created']

# --- MoM Growth (Created) ---
base['mom_growth'] = base['created'].pct_change()

# --- New User Funded Rate ---
new_users = (
    df[df['experience'] == 'New']
    .groupby(['month', 'event_name']).size()
    .unstack(fill_value=0)
)
base['new_funded_rate'] = new_users['Transfer Funded'] / new_users['Transfer Created']

# --- Existing User Funded Rate ---
existing_users = (
    df[df['experience'] == 'Existing']
    .groupby(['month', 'event_name']).size()
    .unstack(fill_value=0)
)
base['existing_funded_rate'] = existing_users['Transfer Funded'] / existing_users['Transfer Created']

# --- New vs Existing Gap ---
base['new_vs_existing_gap'] = base['existing_funded_rate'] - base['new_funded_rate']

# --- Platform Funded Rate (one column per platform) ---
platform = (
    df.groupby(['month', 'platform', 'event_name']).size()
    .unstack(fill_value=0)
)
for p in df['platform'].unique():
    col = f'funded_rate_{p.lower()}'
    base[col] = platform.loc[(slice(None), p), 'Transfer Funded'].values / platform.loc[(slice(None), p), 'Transfer Created'].values

base.round(2)


C:\Users\g-mac\AppData\Local\Temp\ipykernel_28708\1664244945.py:45: RuntimeWarning:

invalid value encountered in divide



event_name,created,funded,transferred,funded_rate,end_to_end_rate,mom_growth,new_funded_rate,existing_funded_rate,new_vs_existing_gap,funded_rate_ios,funded_rate_web,funded_rate_android
month,,,,,,,,,,,,
2024-01,17470,8660,4666,0.50,0.27,NaN,0.41,0.62,0.20,0.57,0.42,0.49
2024-02,25600,10630,6404,0.42,0.25,0.47,0.31,0.59,0.29,0.49,0.38,0.37
2024-03,0,0,10,NaN,inf,-1.00,NaN,NaN,NaN,NaN,NaN,NaN
